# Notebook vs. Script vs. Service

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/09-notebook-vs-script-vs-service.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic fastapi pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

A data scientist spends two weeks building a document summarizer in a Jupyter notebook. The notebook works. It calls the API, processes PDFs, and produces clean summaries. The VP of Engineering asks: "Can we put this in the product?"

The answer takes four more weeks. The notebook has to be rewritten as a service. Half of the logic is in random cells that were run in a non-linear order. There are no tests. Error handling was never added because re-running a cell was easier. The API key is hardcoded. There is no way to call it without opening Jupyter.

This is the notebook-to-production gap, and it kills AI demos regularly. The problem is not that notebooks are bad. It is that the team used a...

## The Concept

### The Three Formats

```
FORMAT        PRIMARY USE             EXPIRES WHEN...
-----------   ---------------------   ----------------------------------
Notebook      Exploration, demos,     You need to run it on a schedule,
              stakeholder review      share it as an API, or run it
                                      more than once without opening
                                      Jupyter

Script        Repeatable pipeline,    More than one user needs to call
              scheduled jobs,         it simultaneously, or it needs to
              CLI tools               stay aliv...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 09 - Notebook vs. Script vs. Service
All three delivery formats for the same AI task in one file.

Run as a script:
    echo "Your text here" | python main.py

Run as a service:
    uvicorn main:app --port 8000
    curl -X POST http://localhost:8000/summarize -H "Content-Type: application/json" -d '{"text": "..."}'
"""

import anthropic
import os
import sys

# ─────────────────────────────────────────────
# SHARED: Core logic (same in all three formats)
# ─────────────────────────────────────────────

def get_client() -> anthropic.Anthropic:
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise EnvironmentError("ANTHROPIC_API_KEY not set")
    return anthropic.Anthropic(api_key=key)


def summarize_text(client: anthropic.Anthropic, text: str) -> str:
    """Call the model and return a one-sentence summary."""
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=128,
        messages=[
            {
                "role": "user",
                "content": f"Summarize the following in one sentence:\n\n{text.strip()}"
            }
        ]
    )
    return response.content[0].text


# ─────────────────────────────────────────────
# FORMAT 2: Script (run directly, stdin input)
# ─────────────────────────────────────────────

def run_as_script() -> None:
    """Entry point when running: echo 'text' | python main.py"""
    text = sys.stdin.read().strip()
    if not text:
        print("Usage: echo 'your text' | python main.py", file=sys.stderr)
        sys.exit(1)

    client = get_client()
    summary = summarize_text(client, text)
    print(summary)


# ─────────────────────────────────────────────
# FORMAT 3: Service (FastAPI HTTP endpoint)
# Only loaded when running with uvicorn
# ─────────────────────────────────────────────

try:
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel

    app = FastAPI(title="Summarizer Service")
    _client = get_client() if os.environ.get("ANTHROPIC_API_KEY") else None

    class SummarizeRequest(BaseModel):
        text: str

    class SummarizeResponse(BaseModel):
        summary: str

    @app.post("/summarize", response_model=SummarizeResponse)
    async def summarize_endpoint(req: SummarizeRequest) -> SummarizeResponse:
        if not req.text.strip():
            raise HTTPException(status_code=422, detail="text cannot be empty")
        if _client is None:
            raise HTTPException(status_code=500, detail="ANTHROPIC_API_KEY not configured")
        summary = summarize_text(_client, req.text)
        return SummarizeResponse(summary=summary)

    @app.get("/health")
    async def health():
        return {"status": "ok"}

except ImportError:
    # FastAPI not installed; that is fine if running as a script
    app = None  # type: ignore


# ─────────────────────────────────────────────
# FORMAT 1: Notebook equivalent (inline demo)
# Shows what the notebook cells look like flattened
# ─────────────────────────────────────────────

def run_notebook_demo() -> None:
    """Simulate what a notebook session looks like, inline."""
    print("=== Notebook-style execution ===")
    print("(In a real notebook, each block below is a separate cell)")
    print()

    # Cell 1: Setup
    print("[Cell 1] Imports and client")
    client = get_client()
    print("  client ready")
    print()

    # Cell 2: Data
    print("[Cell 2] Input text")
    text = (
        "The transformer architecture, introduced in 2017, replaced recurrence "
        "with self-attention. This enabled parallel training across tokens, "
        "which unlocked much larger models and datasets. By 2020, these models "
        "generalized across tasks without task-specific fine-tuning."
    )
    print(f"  {text[:80]}...")
    print()

    # Cell 3: API call
    print("[Cell 3] Call the model")
    summary = summarize_text(client, text)
    print(f"  Summary: {summary}")
    print()
    print("Problem: to run this 'again tomorrow', you reopen Jupyter and")
    print("re-run cells in the right order. There is no main(), no CLI, no HTTP.")

### Demo

In [ ]:
import argparse

parser = argparse.ArgumentParser(description="Summarizer in 3 formats")
parser.add_argument(
    "--demo",
    choices=["notebook", "script"],
    default="script",
    help="notebook: inline demo of notebook-style execution; script: read from stdin"
)
args = parser.parse_args()

if args.demo == "notebook":
    run_notebook_demo()
else:
    run_as_script()

---

## Self-Check

Answer these without running code first:

**1. What is the most accurate description of the work needed?**

- A. Minor: just export the notebook as a Python file and import it.
- B. Significant: the notebook needs to be rewritten as a service with an HTTP interface, error handling, and runtime secret injection.
- C. None: Node.js can execute Jupyter notebooks directly via a subprocess call.
- D. Minor: add a REST framework as a new cell at the bottom of the notebook.

**2. Which promotion is appropriate, and what is the primary trigger?**

- A. No change needed; the script can handle 12 concurrent users.
- B. Promote to service: multiple users need simultaneous access via HTTP.
- C. Promote to notebook: product teams prefer Jupyter for demos.
- D. Promote to script: add argparse for multi-user support.

**3. What is the risk of skipping the notebook/exploration phase here?**

- A. There is no risk; writing a service from scratch is always the correct approach.
- B. They may build a complete service around a prompt approach that turns out to produce poor results, requiring a full rewrite after they discover this.
- C. FastAPI does not support Claude API calls, so the service will fail at runtime.
- D. Services are harder to debug than notebooks, so bugs will be harder to find.

**4. What is the appropriate format for this use case?**

- A. Service: cron jobs require an HTTP endpoint to trigger.
- B. Notebook: GitHub Actions supports Jupyter notebooks natively.
- C. Script: it can be called from a shell command in the workflow YAML.
- D. Service: scheduled jobs always need persistent state between runs.

**5. What does this failure reveal about the notebook?**

- A. Python 3 has a bug with exported notebooks.
- B. The notebook has hidden state: cells were run in a non-linear order that is not reproducible by executing top to bottom.
- C. The export function is broken and should not be used.
- D. The variable naming convention is wrong and needs to be fixed.

**6. What does a /health endpoint provide that a process check does NOT?**

- A. Nothing; process checks and health endpoints are equivalent.
- B. A health endpoint can verify that the API key is valid and the model is reachable, not just that the Python process started.
- C. Health endpoints are required by HTTP spec; process checks are not valid for web services.
- D. A health endpoint prevents the server from crashing under load.

**7. Which question is the MOST diagnostic for production readiness?**

- A. How many parameters does the Claude model have?
- B. Can a new engineer run this from scratch with only a README and an API key, without opening Jupyter?
- C. Is the notebook saved with output cells so we can see the last run?
- D. Does the notebook use the latest version of the anthropic SDK?

_Answers are in `checks.json` in the lesson directory._